# 0505 · 01 · Ingesta de ficheros JSON

En esta demostración vas a ingerir ficheros JSON y a aplicar las transformaciones básicas propias de JSON, como decodificar campos codificados y aplanar textos JSON anidados. Trabajarás con eventos de Kafka simulados, guardados como ficheros JSON en el almacenamiento.

## Objetivos de aprendizaje

Al terminar esta lección serás capaz de:

- Ingerir datos JSON en bruto en Unity Catalog con CTAS y `read_files()`.
- Aplicar varias técnicas para aplanar columnas de texto JSON, con y sin convertirlas a STRUCT.
- Entender la diferencia entre `explode()` y `explode_outer()`.
- Conocer las capacidades y los usos del tipo de datos VARIANT.

## Antes de empezar

1. Ejecuta **`00_preparar_entorno`** (Run all). Deja en `/Volumes/lab0505/upload/upload/events-kafka` dos ficheros con **2.252 eventos**.
2. Conecta este notebook a **Serverless**. Para la sección D (VARIANT), el entorno de Serverless debe ser la **versión 2 o superior**: lo ves en el panel **Environment**, a la derecha del notebook.

Todas las tablas del laboratorio se crean en el schema `lab0505.bronze`:

In [ ]:
%sql
USE CATALOG lab0505;
USE SCHEMA bronze;

SELECT current_catalog() AS catalogo, current_schema() AS schema;

## B. CTAS con `read_files()` para ingerir ficheros JSON

### B1. Inspeccionar los ficheros JSON

1. Ejecuta la celda siguiente para comprobar que hay **2 ficheros JSON** en `/Volumes/lab0505/upload/upload/events-kafka`.

In [ ]:
%sql
-- Listar los ficheros del volumen
LIST '/Volumes/lab0505/upload/upload/events-kafka';

2. Ejecuta la celda de abajo para ver los datos JSON en bruto. Fíjate en lo siguiente:
   - Cada fila contiene un JSON con 6 pares clave/valor: es el "sobre" con el que Kafka entrega cada mensaje.
   - Los campos **key** y **value** están codificados en Base64. Base64 es un sistema que convierte datos binarios en un texto formado por 64 caracteres imprimibles, para poder guardarlos o enviarlos como texto sin que se corrompan.

**Ejemplo de salida, formateado** (el primer evento de `000.json`):

```json
{
    "key": "VUEwMDAwMDAwMDAwMDAwMA==",
    "offset": 0,
    "partition": 0,
    "timestamp": 1596240000000,
    "topic": "clickstream",
    "value": "eyJkZXZpY2UiOiJDaHJvbWUiLCJlY29tbWVyY2UiOnsicHVyY2hhc2VfcmV2ZW51ZV9pbl91c2QiOjg1MC41LCJ0b3RhbF9pdGVtX3F1YW50aXR5IjoxLCJ1bmlxdWVfaXRlbXMiOjF9LCJldmVudF9uYW1lIjoiZmluYWxpemUiLCJldmVudF9wcmV2aW91c190aW1lc3RhbXAiOm51bGwsImV2ZW50X3RpbWVzdGFtcCI6MTU5NjI0MDAwMDAwMDAwMCwiZ2VvIjp7ImNpdHkiOiJNYWRyaWQiLCJzdGF0ZSI6Ik1EIn0sIml0ZW1zIjpbeyJjb3Vwb24iOiJORVdCRUQxMCIsIml0ZW1faWQiOiJNX1NUQU5fVCIsIml0ZW1fbmFtZSI6IlN0YW5kYXJkIFR3aW4gTWF0dHJlc3MiLCJpdGVtX3JldmVudWVfaW5fdXNkIjo4NTAuNSwicHJpY2VfaW5fdXNkIjo4NTAuNSwicXVhbnRpdHkiOjF9XSwidHJhZmZpY19zb3VyY2UiOiJlbWFpbCIsInVzZXJfZmlyc3RfdG91Y2hfdGltZXN0YW1wIjoxNTk2MjQwMDAwMDAwMDAwLCJ1c2VyX2lkIjoiVUEwMDAwMDAwMDAwMDAwMCJ9"
}
```

In [ ]:
%sql
-- Ver los ficheros JSON como texto
SELECT *
FROM text.`/Volumes/lab0505/upload/upload/events-kafka`
LIMIT 5;

3. Ejecuta la celda siguiente para leer los ficheros JSON con `read_files()`. Fíjate en que cada clave del JSON se convierte en su propia columna, más `_rescued_data`. Con JSON no hace falta indicar separador ni cabecera, a diferencia del CSV del laboratorio 0504: los nombres van dentro de cada registro.

In [ ]:
%sql
-- Ver los ficheros JSON con read_files
SELECT *
FROM read_files(
  "/Volumes/lab0505/upload/upload/events-kafka",
  format => "json"
)
LIMIT 10;

### B2. CTAS y `read_files()` con JSON

Ingerir ficheros JSON con `read_files()` es tan sencillo como leer CSV.

1. Ejecuta la celda de abajo para guardar estos datos en bruto en la tabla **kafka_events_bronze_raw** y verla. Al revisar el resultado, fíjate en lo siguiente:
   - Las columnas **key** y **value** son de tipo STRING y contienen datos **codificados en Base64**.
   - Es decir, el contenido real se ha codificado en Base64 y se ha guardado como texto.
   - En esta primera tabla Bronze todavía no se han convertido en texto legible.

**NOTA:** la codificación Base64 es habitual al ingerir datos de colas de mensajes o plataformas de streaming (como Kafka), donde importa transportar datos binarios de forma segura y sin corromperlos.

In [ ]:
%sql
-- Crear la tabla Delta
CREATE OR REPLACE TABLE kafka_events_bronze_raw AS
SELECT *
FROM read_files(
  "/Volumes/lab0505/upload/upload/events-kafka",
  format => "json"
);

-- Ver la tabla
SELECT *
FROM kafka_events_bronze_raw
LIMIT 10;

   Comprueba que la tabla tiene **2.252 eventos**:

In [ ]:
%sql
SELECT count(*) AS eventos
FROM kafka_events_bronze_raw;

### B3. Decodificar los textos Base64 para la tabla Bronze

1. Decodifica las columnas **key** y **value** y fíjate en sus tipos tras aplicar la función [`unbase64()`](https://docs.databricks.com/aws/en/sql/language-manual/functions/unbase64), que devuelve el Base64 decodificado como binario:
   - **encoded_key**: la columna **key** original, en Base64.
   - **decoded_key**: una columna nueva con **key** decodificada de Base64 a BINARY.
   - **encoded_value**: la columna **value** original, en Base64.
   - **decoded_value**: una columna nueva con **value** decodificada de Base64 a BINARY.

   Ejecuta la celda y revisa el resultado. Fíjate en que **decoded_key** y **decoded_value** son de tipo BINARY: todavía no se leen como texto.

In [ ]:
%sql
-- Decodificar el Base64
SELECT
  key AS encoded_key,
  unbase64(key) AS decoded_key,
  value AS encoded_value,
  unbase64(value) AS decoded_value
FROM kafka_events_bronze_raw
LIMIT 5;

2. Ejecuta la celda siguiente para convertir las columnas BINARY en STRING con `CAST`. Fíjate en lo siguiente:
   - **decoded_key** y **decoded_value** son ahora de tipo STRING y se leen: la clave es un identificador como `UA00000000000000`, el mismo valor que el `user_id` del evento.
   - **decoded_value** es un texto en formato JSON.

In [ ]:
%sql
-- Convertir el BINARY en STRING con CAST
SELECT
  key AS encoded_key,
  cast(unbase64(key) AS STRING) AS decoded_key,
  value AS encoded_value,
  cast(unbase64(value) AS STRING) AS decoded_value
FROM kafka_events_bronze_raw
LIMIT 5;

3. Ahora júntalo todo para crear otra tabla Bronze llamada **kafka_events_bronze_decoded**, con las columnas **key** y **value** de **kafka_events_bronze_raw** ya decodificadas. Tendrá las mismas 2.252 filas: decodificar no añade ni quita eventos.

In [ ]:
%sql
-- Crear la tabla bronze_decoded
CREATE OR REPLACE TABLE kafka_events_bronze_decoded AS
SELECT
  cast(unbase64(key) AS STRING) AS decoded_key,
  offset,
  partition,
  timestamp,
  topic,
  cast(unbase64(value) AS STRING) AS decoded_value
FROM kafka_events_bronze_raw;

-- Ver la tabla nueva
SELECT *
FROM kafka_events_bronze_decoded
LIMIT 5;

## C. Trabajar con textos JSON en una tabla

### C1. Aplanar columnas de texto JSON

Ahora verás cómo extraer una columna de otra que contiene un texto en formato JSON.

**VENTAJAS**

- **Sencillo**: fácil de implementar; el JSON se guarda como texto plano.
- **Flexible**: admite cualquier estructura JSON, sin restricciones de esquema.

**CONSIDERACIONES**

- **Rendimiento**: las columnas STRING son más lentas al consultar y procesar datos complejos.
- **Sin esquema**: al no haber un esquema definido, pueden aparecer problemas de integridad de datos.
- **Difícil de consultar**: hace falta código adicional para interpretar y extraer los datos.

#### C1.1 Consultar textos JSON

Puedes extraer una columna de un campo que contiene texto JSON con la sintaxis `<nombre-columna>:<ruta-de-extracción>`, donde `<nombre-columna>` es la columna de texto y `<ruta-de-extracción>` es la ruta del campo que quieres extraer. **El resultado es siempre texto.** Puedes acceder a campos anidados con `.` o `[]`.

Así se aprovecha la funcionalidad integrada de Spark SQL para trabajar directamente con datos anidados guardados como texto JSON.

[Consultar textos JSON](https://docs.databricks.com/aws/en/semi-structured/json)

Ejemplo de texto JSON de una fila de la columna **decoded_value** (una compra):

```json
{
    "device": "Chrome",
    "ecommerce": {
        "purchase_revenue_in_usd": 850.5,
        "total_item_quantity": 1,
        "unique_items": 1
    },
    "event_name": "finalize",
    "event_previous_timestamp": null,
    "event_timestamp": 1596240000000000,
    "geo": {
        "city": "Madrid",
        "state": "MD"
    },
    "items": [
        {
            "coupon": "NEWBED10",
            "item_id": "M_STAN_T",
            "item_name": "Standard Twin Mattress",
            "item_revenue_in_usd": 850.5,
            "price_in_usd": 850.5,
            "quantity": 1
        }
    ],
    "traffic_source": "email",
    "user_first_touch_timestamp": 1596240000000000,
    "user_id": "UA00000000000000"
}
```

La mayoría de eventos no son compras (`page_view`, `view_item`, `add_to_cart`): llegan con `"ecommerce": {}` vacío e `"items": []`, y unos pocos con `"items": null`.

1. Por ejemplo, extrae estos valores del texto JSON:
   - `decoded_value:device`
   - `decoded_value:traffic_source`
   - `decoded_value:geo`
   - `decoded_value:items`

   Ejecuta la celda y revisa el resultado. Has extraído los valores del texto JSON:
   - **device** es un STRING.
   - **traffic_source** es un STRING.
   - **geo** es un STRING que contiene otro texto JSON: para sacar la ciudad hay que volver a aplicar la ruta, `decoded_value:geo.city`.
   - **items** es un STRING que contiene un array de textos JSON.

In [ ]:
%sql
-- Consultar un texto JSON y extraer valores
SELECT
  decoded_value,
  decoded_value:device,
  decoded_value:traffic_source,
  decoded_value:geo,       ----- Contiene otro texto JSON
  decoded_value:items      ----- Contiene un array anidado de textos JSON
FROM kafka_events_bronze_decoded
LIMIT 5;

   Con `LIMIT 5` es probable que `items` salga vacío (`[]`) en todas las filas: solo las compras llevan artículos. Para verlos, filtra las compras:

In [ ]:
%sql
SELECT
  decoded_value:event_name,
  decoded_value:geo.city,
  decoded_value:items
FROM kafka_events_bronze_decoded
WHERE decoded_value:event_name = 'finalize'
LIMIT 5;

2. Ya puedes extraer los valores que necesites para crear otra tabla Bronze que aplane el texto JSON para el procesamiento posterior.

   Las columnas nuevas se llaman **device**, **traffic_source**, **geo** e **items** sin necesidad de alias: Databricks usa el último tramo de la ruta. Las cuatro siguen siendo STRING, y `geo` e `items` todavía contienen JSON: la tabla solo está aplanada en el primer nivel.

In [ ]:
%sql
-- Aplanar el texto JSON
CREATE OR REPLACE TABLE kafka_events_bronze_string_flattened AS
SELECT
  decoded_key,
  offset,
  partition,
  timestamp,
  topic,
  decoded_value:device,
  decoded_value:traffic_source,
  decoded_value:geo,       ----- Contiene otro texto JSON
  decoded_value:items      ----- Contiene un array anidado de textos JSON
FROM kafka_events_bronze_decoded;

-- Ver la tabla
SELECT *
FROM kafka_events_bronze_string_flattened;

### C2. Aplanar textos JSON convirtiéndolos a STRUCT

Igual que en la sección anterior, ahora vas a aplanar la columna de texto JSON **decoded_value**, pero usando una columna STRUCT.

#### Ventajas y consideraciones de las columnas STRUCT

**Ventajas**

- **Esquema garantizado**: las columnas STRUCT definen e imponen un esquema, lo que ayuda a mantener la integridad de los datos.
- **Mejor rendimiento**: los STRUCT suelen ser más eficientes de consultar y procesar que el texto plano.

**Consideraciones**

- **Esquema garantizado**: como el esquema se impone, pueden surgir problemas si la estructura del JSON cambia con el tiempo.
- **Menos flexibilidad**: los datos tienen que encajar siempre con el esquema definido, con poco margen para variaciones.

#### C2.1 Convertir un texto JSON en una columna STRUCT

Para convertir una columna de texto JSON en STRUCT, hay que obtener el esquema del texto JSON y después interpretarlo como tipo STRUCT.

Se hace en dos pasos:

1. Obtener el tipo STRUCT del texto JSON.
2. Aplicar ese STRUCT a la columna de texto JSON.

**NOTA:** en esta demostración ya tienes copiados los valores correctos. La celda del paso 2 lleva pegado el resultado (una sola fila) de ejecutar la celda del paso 1.

1. Obtén el esquema con la función [`schema_of_json()`](https://docs.databricks.com/aws/en/sql/language-manual/functions/schema_of_json), que devuelve el esquema inferido de un texto JSON.

   Ejecuta la celda y revisa el resultado: muestra la estructura del texto JSON.

   > **Por qué esta fila.** `schema_of_json` infiere el esquema de **un solo** texto. Por eso se usa una compra con todos los campos rellenos: si `items` estuviera vacío o `event_previous_timestamp` fuera `null`, el esquema saldría incompleto o con tipos equivocados.

In [ ]:
%sql
-- Obtener el esquema del texto JSON
SELECT schema_of_json('{"device":"Linux","ecommerce":{"purchase_revenue_in_usd":1881.0,"total_item_quantity":2,"unique_items":1},"event_name":"finalize","event_previous_timestamp":1596240138488000,"event_timestamp":1596240190421000,"geo":{"city":"Bilbao","state":"PV"},"items":[{"coupon":"NEWBED10","item_id":"M_STAN_Q","item_name":"Standard Queen Mattress","item_revenue_in_usd":1881.0,"price_in_usd":940.5,"quantity":2}],"traffic_source":"search","user_first_touch_timestamp":1595289600000000,"user_id":"UA00000000000011"}')
AS schema;

2. Copia el resultado de `schema_of_json` en la función [`from_json()`](https://docs.databricks.com/aws/en/sql/language-manual/functions/from_json). Esta función convierte una columna de texto JSON en un tipo STRUCT con el esquema indicado, y crea una tabla nueva llamada **kafka_events_bronze_struct**.

   Ejecuta la celda y revisa el resultado. La columna **value** se ha convertido en un STRUCT anidado con campos simples, structs anidados (`ecommerce`, `geo`) y un array de structs (`items`). Puedes desplegarla en la tabla de resultados.

In [ ]:
%sql
CREATE OR REPLACE TABLE kafka_events_bronze_struct AS
SELECT
  * EXCEPT (decoded_value),
  from_json(
      decoded_value,    -- Columna con el texto JSON
      'STRUCT<device: STRING, ecommerce: STRUCT<purchase_revenue_in_usd: DOUBLE, total_item_quantity: BIGINT, unique_items: BIGINT>, event_name: STRING, event_previous_timestamp: BIGINT, event_timestamp: BIGINT, geo: STRUCT<city: STRING, state: STRING>, items: ARRAY<STRUCT<coupon: STRING, item_id: STRING, item_name: STRING, item_revenue_in_usd: DOUBLE, price_in_usd: DOUBLE, quantity: BIGINT>>, traffic_source: STRING, user_first_touch_timestamp: BIGINT, user_id: STRING>') AS value
FROM kafka_events_bronze_decoded;

-- Ver la tabla nueva
SELECT *
FROM kafka_events_bronze_struct
LIMIT 10;

#### C2.2 Extraer campos, campos anidados y arrays anidados de columnas STRUCT

Puedes consultar la columna STRUCT con `value.device` o `value.ecommerce` en el `SELECT`.

1. Con esta sintaxis obtienes valores de la columna STRUCT **value**. Ejecuta la celda y revisa el resultado. Fíjate en lo siguiente:
   - Has obtenido los valores **device** y **city** de la columna STRUCT. A diferencia de C1, cada campo sale **con su tipo**, no siempre como texto.
   - La columna **items** contiene un ARRAY de STRUCT, y el número de elementos varía:

     | `number_elements_in_array` | Filas | Qué son |
     |---|---|---|
     | 3 | 103 | Compras de 3 artículos |
     | 1 | 197 | Compras de 1 artículo |
     | 0 | 1.902 | Eventos con `items: []` |
     | `NULL` | 50 | Eventos con `items: null` (en orden descendente, al final) |

In [ ]:
%sql
-- Obtener valores de una columna STRUCT
SELECT
  decoded_key,
  value.device as device,  -- <----- Campo
  value.geo.city as city,  -- <----- Campo anidado dentro de geo
  value.items as items,
  array_size(items) AS number_elements_in_array -- <----- Cuenta los elementos del array items
FROM kafka_events_bronze_struct
ORDER BY number_elements_in_array DESC;

#### C2.3 Explotar arrays

Explotar (*explode*) un array convierte cada elemento de una columna array en una fila aparte, aplanando el array. Ten en cuenta lo siguiente al usar esta función:

1. Devuelve un conjunto de filas con los elementos del array, o con las claves y valores si es un map.
2. Si el array es `NULL` o está vacío, no genera ninguna fila. Para obtener una fila con `NULL` en esos casos, usa la función [`explode_outer()`](https://docs.databricks.com/aws/en/sql/language-manual/functions/explode_outer).

1. Ejecuta la celda para ver cómo el ARRAY de `value.items` se explota en una fila por cada elemento.

   Deberías obtener **506 filas**: 103 compras × 3 artículos + 197 compras × 1. Cada compra de 3 artículos aparece 3 veces, con el mismo `decoded_key` y un `item_in_array` distinto. Los 1.952 eventos sin artículos no aparecen.

   **NOTA:** el `ORDER BY` dentro del `CREATE TABLE` no garantiza el orden al consultar la tabla después: una tabla no guarda un orden de filas.

In [ ]:
%sql
-- Explotar el array en una fila por elemento
CREATE OR REPLACE TABLE bronze_explode_array AS
SELECT
  decoded_key,
  array_size(value.items) AS number_elements_in_array,
  explode(value.items) AS item_in_array,
  value.items
FROM kafka_events_bronze_struct
ORDER BY number_elements_in_array DESC;

-- Ver la tabla
SELECT *
FROM bronze_explode_array;

2. Compara los recuentos de `explode()` y `explode_outer()`:
   - `explode()`: **506** filas. Solo los artículos.
   - `explode_outer()`: **2.458** filas. Los 506 artículos más una fila con `NULL` por cada uno de los 1.952 eventos sin artículos.

In [ ]:
%sql
SELECT
  (SELECT count(*) FROM (SELECT explode(value.items) FROM kafka_events_bronze_struct))       AS filas_explode,
  (SELECT count(*) FROM (SELECT explode_outer(value.items) FROM kafka_events_bronze_struct)) AS filas_explode_outer;

## D. Trabajar con una columna VARIANT

**Ventajas y consideraciones de las columnas VARIANT:**

**VENTAJAS**

- **Abierto**: de código abierto, sin dependencia de un formato propietario.
- **Flexible**: sin esquema estricto. Puedes guardar en VARIANT cualquier tipo de dato semiestructurado.
- **Rápido**: mejor rendimiento que los métodos anteriores.

**CONSIDERACIONES**

- Disponible en Databricks Runtime 15.4 y superior.
- [Soporte de VARIANT en Delta Lake](https://docs.databricks.com/aws/en/tables/features/variant)

**RECURSOS:**

- [El tipo VARIANT (documentación)](https://docs.databricks.com/aws/en/semi-structured/variant)
- [Introducing the Open Variant Data Type in Delta Lake and Apache Spark](https://www.databricks.com/blog/introducing-open-variant-data-type-delta-lake-and-apache-spark)

**NOTA:** el tipo VARIANT no funciona en la versión 1 del entorno Serverless. Compruébalo en el panel **Environment** del notebook.

1. Consulta la tabla **kafka_events_bronze_decoded**. Confirma que la columna **decoded_value** contiene un texto en formato JSON.

In [ ]:
%sql
SELECT *
FROM kafka_events_bronze_decoded
LIMIT 5;

2. Usa la función [`parse_json`](https://docs.databricks.com/aws/en/sql/language-manual/functions/parse_json) para obtener un valor VARIANT a partir del texto JSON.

   Ejecuta la celda y revisa el resultado. La columna **json_variant_value** es de tipo VARIANT. A diferencia de C2, **no hace falta calcular ni pegar ningún esquema**: VARIANT guarda el JSON con su estructura, sea cual sea.

   `parse_json` falla si algún texto no es JSON válido; la alternativa tolerante es `try_parse_json`, que devuelve `NULL`.

In [ ]:
%sql
-- Crear una columna VARIANT
CREATE OR REPLACE TABLE kafka_events_bronze_variant AS
SELECT
  decoded_key,
  offset,
  partition,
  timestamp,
  topic,
  parse_json(decoded_value) AS json_variant_value   -- Convierte decoded_value al tipo VARIANT
FROM kafka_events_bronze_decoded;

-- Ver la tabla
SELECT *
FROM kafka_events_bronze_variant
LIMIT 5;

3. Puedes consultar la columna VARIANT con `:` para construir la tabla que necesites.

   [El tipo VARIANT](https://docs.databricks.com/aws/en/semi-structured/variant)

   Sobre VARIANT, `:` devuelve otro **VARIANT**; con `::` lo conviertes al tipo que quieras. Por eso `device` lleva `:: STRING` e `items` no: se queda como VARIANT y conserva su estructura de array, que puedes seguir recorriendo (`json_variant_value:items[0].item_name`).

   | | Sintaxis | Qué devuelve |
   |---|---|---|
   | **C1 · STRING** | `decoded_value:device` | Siempre STRING |
   | **C2 · STRUCT** | `value.device` | El tipo del esquema que pegaste |
   | **D · VARIANT** | `json_variant_value:device` | VARIANT; con `:: STRING` lo conviertes |

In [ ]:
%sql
SELECT
  json_variant_value,
  json_variant_value:device :: STRING,  -- Obtiene device y lo convierte a STRING
  json_variant_value:items
FROM kafka_events_bronze_variant
LIMIT 10;

## Extensión opcional: el JSON cambia

La consideración más importante de STRUCT era: *"pueden surgir problemas si la estructura del JSON cambia con el tiempo"*. Aquí lo compruebas.

`00_preparar_entorno` dejó en `/Volumes/lab0505/upload/upload/events-kafka-extra` dos ficheros más recientes, `006.json` y `007.json`. Algunos de sus eventos traen un campo nuevo, **`campaign`**, que no estaba en los datos originales ni en el esquema STRUCT que pegaste.

1. Cuenta cuántos eventos traen `campaign`, leyendo el texto JSON. Deberías ver **367** de 1.100.

In [ ]:
%sql
WITH eventos AS (
  SELECT cast(unbase64(value) AS STRING) AS decoded_value
  FROM read_files("/Volumes/lab0505/upload/upload/events-kafka-extra", format => "json")
)
SELECT
  count(*) AS eventos,
  count(decoded_value:campaign) AS con_campaign
FROM eventos;

2. Léelos con el esquema STRUCT de C2 y con VARIANT, y compara:
   - Con **STRUCT**, `campaign` no existe: el esquema fijo descarta el campo sin avisar. Para verlo habría que ampliar el esquema.
   - Con **VARIANT**, `campaign` está ahí sin tocar nada, por ejemplo `{"channel":"social","experiment":"layout-b","id":"campaign-6"}`.

In [ ]:
%sql
WITH eventos AS (
  SELECT cast(unbase64(value) AS STRING) AS decoded_value
  FROM read_files("/Volumes/lab0505/upload/upload/events-kafka-extra", format => "json")
)
SELECT
  from_json(decoded_value, 'STRUCT<device: STRING, ecommerce: STRUCT<purchase_revenue_in_usd: DOUBLE, total_item_quantity: BIGINT, unique_items: BIGINT>, event_name: STRING, event_previous_timestamp: BIGINT, event_timestamp: BIGINT, geo: STRUCT<city: STRING, state: STRING>, items: ARRAY<STRUCT<coupon: STRING, item_id: STRING, item_name: STRING, item_revenue_in_usd: DOUBLE, price_in_usd: DOUBLE, quantity: BIGINT>>, traffic_source: STRING, user_first_touch_timestamp: BIGINT, user_id: STRING>') AS como_struct,
  parse_json(decoded_value):campaign AS campaign_en_variant
FROM eventos
WHERE decoded_value:campaign IS NOT NULL
LIMIT 5;

## Comprueba lo que has aprendido

1. **¿Por qué hay que usar `unbase64` y además `cast`?** `unbase64` devuelve BINARY; el `cast(… AS STRING)` lo convierte en texto legible.
2. **¿Qué diferencia hay entre `decoded_value:device` y `value.device`?** El primero extrae de un texto JSON y siempre devuelve STRING; el segundo lee un campo de un STRUCT con su tipo.
3. **¿Por qué `schema_of_json` se aplica a una compra y no a cualquier evento?** Porque infiere el esquema de un solo texto: con `items` vacío o campos a `null`, el esquema saldría incompleto o con tipos equivocados.
4. **¿Qué diferencia hay entre `explode()` y `explode_outer()`?** `explode()` no genera filas para arrays vacíos o nulos (506 filas); `explode_outer()` genera una fila con `NULL` para cada uno (2.458).
5. **¿Cuándo elegirías STRING, STRUCT o VARIANT?** STRING si solo guardas y consultas poco; STRUCT si el esquema es estable y quieres tipos y rendimiento; VARIANT si el JSON cambia o no conoces su forma, sin renunciar al rendimiento.

## Volver a empezar

Ejecuta `00_preparar_entorno`: borra las tablas y deja los ficheros como al principio. Las tablas no se borran al terminar este notebook, para que puedas seguir inspeccionándolas.

## Recursos adicionales

- [read_files](https://docs.databricks.com/aws/en/sql/language-manual/functions/read_files)
- [Consultar textos JSON](https://docs.databricks.com/aws/en/semi-structured/json)
- [from_json](https://docs.databricks.com/aws/en/sql/language-manual/functions/from_json) y [schema_of_json](https://docs.databricks.com/aws/en/sql/language-manual/functions/schema_of_json)
- [explode_outer](https://docs.databricks.com/aws/en/sql/language-manual/functions/explode_outer)
- [El tipo VARIANT](https://docs.databricks.com/aws/en/semi-structured/variant)